## Multi-Representation Indexing

In [5]:
import uuid
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import MultiVectorRetriever

In [2]:
# Load the Raw Documents 

print("--- 1. Loading Documents ---")

loader1 = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader1.load()

loader2 = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader2.load())

print(f"Total documents loaded: {len(docs)}")
print(f"Type of documents: {type(docs[0])}\n")

--- 1. Loading Documents ---
Total documents loaded: 2
Type of documents: <class 'langchain_core.documents.base.Document'>



In [3]:
# Generate Summaries (The "Bait")

print("--- 2. Generating Summaries ---")

# We use a simple chain to ask the LLM to summarize the page content
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document concisely:\n\n{doc}")
    | ChatOpenAI(model="gpt-4o-mini", max_retries=0)
    | StrOutputParser()
)

# .batch() processes all documents in parallel
summaries = chain.batch(docs, {"max_concurrency": 5})

print(f"Generated {len(summaries)} summaries.")
print(f"Type of summary output: {type(summaries[0])}")
print(f"Preview of first summary: {summaries[0][:150]}...\n")

--- 2. Generating Summaries ---
Generated 2 summaries.
Type of summary output: <class 'langchain_core.messages.base.TextAccessor'>
Preview of first summary: The document by Lilian Weng explores the development of LLM (Large Language Model)-powered autonomous agents, outlining their essential components: pl...



In [6]:
# Setup the Databases and Retriever 

print("--- 3. Setting up Multi-Vector Retriever ---")

# 3a. Vectorstore (To hold the embeddings of the summaries)
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)

# 3b. Docstore (To hold the raw, full-text parent documents)
# Note: Replaced InMemoryByteStore with InMemoryStore to handle Document objects directly
store = InMemoryStore()
id_key = "doc_id"

# 3c. The Retriever that links them together
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

--- 3. Setting up Multi-Vector Retriever ---


In [8]:
# Link Summaries to Parent Docs and Index them

print("--- 4. Linking and Indexing ---")

# Generate a unique ID for each document
doc_ids = [str(uuid.uuid4()) for _ in docs]
print(f"Generated Document IDs: {doc_ids}")

# Create Document objects for the summaries, embedding the parent ID in the metadata
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

print(f"Type of summary_docs: {type(summary_docs[0])}")
print(f"Metadata injected into summary: {summary_docs[0].metadata}")

# Add the summaries to the Vector Database
retriever.vectorstore.add_documents(summary_docs)

# Add the full original documents to the Document Store (mapped to the same IDs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

print("Successfully added summaries to Vectorstore and full docs to Docstore.\n")

--- 4. Linking and Indexing ---
Generated Document IDs: ['334998e4-88dd-422f-9f6e-ab690a2c87c7', '05d9f685-bc4d-4b53-90b4-de780535b4c1']
Type of summary_docs: <class 'langchain_core.documents.base.Document'>
Metadata injected into summary: {'doc_id': '334998e4-88dd-422f-9f6e-ab690a2c87c7'}
Successfully added summaries to Vectorstore and full docs to Docstore.



In [9]:
# Execution and Comparison 

print("--- 5. Retrieval Comparison ---")
query = "Memory in agents"

# A. Direct Vectorstore Search (This only hits the summaries)
print("\n>>> A. Direct Vectorstore Search (Hits Summary Only):")
sub_docs = vectorstore.similarity_search(query, k=1)

print(f"Type returned: {type(sub_docs[0])}")
print(f"Metadata: {sub_docs[0].metadata}")
print(f"Content length: {len(sub_docs[0].page_content)} characters")
print(f"Content Preview:\n{sub_docs[0].page_content[:300]}...\n")

# B. Multi-Vector Retriever Search (The "Switch" - Returns the Parent Doc)
# Note: Modern LangChain uses .invoke() instead of .get_relevant_documents()
print("\n>>> B. Multi-Vector Retriever Search (Returns Full Parent Document):")
retrieved_docs = retriever.invoke(query)

# We slice the list because the Retriever might return multiple docs based on its configuration, 
# but since we only have 2 docs in the DB, it will likely return the best one.
best_doc = retrieved_docs[0]

print(f"Type returned: {type(best_doc)}")
print(f"Metadata: {best_doc.metadata}") # Notice the doc_id is NOT here, this is the original metadata!
print(f"Content length: {len(best_doc.page_content)} characters")
print(f"Content Preview:\n{best_doc.page_content[:300]}...\n")

--- 5. Retrieval Comparison ---

>>> A. Direct Vectorstore Search (Hits Summary Only):
Type returned: <class 'langchain_core.documents.base.Document'>
Metadata: {'doc_id': '334998e4-88dd-422f-9f6e-ab690a2c87c7'}
Content length: 1530 characters
Content Preview:
The document by Lilian Weng explores the development of LLM (Large Language Model)-powered autonomous agents, outlining their essential components: planning, memory, and tool use. 

1. **Agent System Overview**: LLM serves as the brain for these agents, enabling them to decompose complex tasks into ...


>>> B. Multi-Vector Retriever Search (Returns Full Parent Document):
Type returned: <class 'langchain_core.documents.base.Document'>
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer an